In [6]:
import sys
sys.path.append('../..')

import pandas as pd
from utils.db_utils import write_table, read_table

In [7]:
date = read_table("select * from sc_gold.dim_date")
state= read_table("select * from sc_gold.dim_state")
qualification = read_table("select * from sc_gold.dim_qualification")

In [8]:
df = read_table("select * from sc_bronze.dosm_graduates_state")
df



,state,year,qualification,emp_graduate,outside_labour_force,total_graduate,unemp_graduate,unemp_rate
0,Johor,2016,degree,143700.0,38900.0,190800.0,8200.0,5.4
1,Johor,2016,diploma,152500.0,17200.0,177200.0,7500.0,4.7
2,Johor,2017,degree,163900.0,45600.0,216700.0,7200.0,4.2
3,Johor,2017,diploma,171800.0,21100.0,201200.0,8300.0,4.6
4,Johor,2018,degree,190500.0,51800.0,250500.0,8200.0,4.1
...,...,...,...,...,...,...,...,...
283,Terengganu,2022,diploma,59100.0,22800.0,85000.0,3000.0,4.9
284,Terengganu,2023,degree,62000.0,14800.0,79200.0,2300.0,3.6
285,Terengganu,2023,diploma,61500.0,24400.0,88500.0,2700.0,4.1
286,Terengganu,2024,degree,67400.0,13200.0,83400.0,2800.0,4.0


In [9]:
df["date"] = pd.to_datetime(df["year"], format="%Y")
df = df.merge(
    date[["date", "date_id"]],
    on="date",
    how="left"
)

df = df.merge(
    state[["state", "state_id"]],
    on="state",
    how="left"
)

df = df.merge(
    qualification[["qualification", "qualification_id"]],
    on="qualification",
    how="left"
)


df_final = df.drop(columns=["year", "date", "state", "qualification"])
id_cols = ["date_id", "state_id", "qualification_id"]
df_final = df_final[id_cols + [col for col in df_final.columns if col not in id_cols]]

In [10]:
df_final["grad_id"] = ["GRAD" + str(i+1).zfill(4) for i in range(len(df_final))]
df_final = df_final[["grad_id"] + [c for c in df_final.columns if c != "grad_id"]]
df_final

,grad_id,date_id,state_id,qualification_id,emp_graduate,outside_labour_force,total_graduate,unemp_graduate,unemp_rate
0,GRAD0001,DT001,ST001,Q001,143700.0,38900.0,190800.0,8200.0,5.4
1,GRAD0002,DT001,ST001,Q002,152500.0,17200.0,177200.0,7500.0,4.7
2,GRAD0003,DT005,ST001,Q001,163900.0,45600.0,216700.0,7200.0,4.2
3,GRAD0004,DT005,ST001,Q002,171800.0,21100.0,201200.0,8300.0,4.6
4,GRAD0005,DT009,ST001,Q001,190500.0,51800.0,250500.0,8200.0,4.1
...,...,...,...,...,...,...,...,...,...
283,GRAD0284,DT025,ST013,Q002,59100.0,22800.0,85000.0,3000.0,4.9
284,GRAD0285,DT029,ST013,Q001,62000.0,14800.0,79200.0,2300.0,3.6
285,GRAD0286,DT029,ST013,Q002,61500.0,24400.0,88500.0,2700.0,4.1
286,GRAD0287,DT033,ST013,Q001,67400.0,13200.0,83400.0,2800.0,4.0


In [11]:
write_table(df_final, "sc_gold", "fact_graduates")

Table sc_gold.fact_graduates written successfully.
